# CLA RAG Pipeline Comparison

This notebook compares all 5 RAG pipelines on the Complex Lymphatic Anomaly (CLA) dataset:

| Pipeline | Strategy |
|----------|----------|
| **01 Basic RAG** | Dense vector retrieval baseline |
| **02 KG-RAG** | Knowledge graph + vector hybrid |
| **03 HyDE RAG** | Hypothetical document embeddings |
| **04 Self-RAG** | Self-reflective adaptive retrieval |
| **05 Multi-hop RAG** | Iterative chain-of-thought retrieval |

**Case study:** Complex Lymphatic Anomalies including GSD, GLA, KLA, CCLA, and LAM.

In [ ]:
import os
import sys
import json
import time
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Configure OpenAI key
# os.environ['OPENAI_API_KEY'] = 'your-key-here'

CONFIG_PATH = '../config/config.yaml'
DATASET_PATH = '../data/pseudo_dataset/cla_documents.json'
EVAL_PATH = '../data/pseudo_dataset/eval_questions.json'

print('Environment ready.')

## 1. Generate the Pseudo Dataset

In [ ]:
from pathlib import Path

if not Path(DATASET_PATH).exists():
    print('Generating dataset...')
    import subprocess
    result = subprocess.run(
        [sys.executable, '../data/pseudo_dataset/generate_dataset.py'],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)
else:
    print(f'Dataset already exists at {DATASET_PATH}')

with open(DATASET_PATH) as f:
    dataset = json.load(f)

docs = dataset['documents']
print(f'\nDataset: {len(docs)} documents')
print(f'Disease subtypes: {set(d["disease_entity"] for d in docs)}')
print(f'Source types: {set(d["source_type"] for d in docs)}')

## 2. Document Corpus Overview

In [ ]:
df_docs = pd.DataFrame([
    {
        'id': d['id'],
        'title': d['title'][:60] + '...',
        'disease': d['disease_entity'],
        'type': d['source_type'],
        'year': d['year'],
        'journal': d['journal'],
        'text_length': len(d.get('full_text', '')),
    }
    for d in docs
])

print(df_docs[['id', 'disease', 'type', 'year', 'text_length']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Disease distribution
disease_counts = df_docs['disease'].value_counts()
axes[0].bar(disease_counts.index, disease_counts.values, color=['#4CAF50','#2196F3','#FF9800','#9C27B0','#F44336'])
axes[0].set_title('Documents by Disease Subtype')
axes[0].set_xlabel('Disease')
axes[0].tick_params(axis='x', rotation=30)

# Source type distribution
type_counts = df_docs['type'].value_counts()
axes[1].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Documents by Source Type')

# Text length distribution
axes[2].hist(df_docs['text_length'], bins=10, color='steelblue', edgecolor='white')
axes[2].set_title('Document Text Length Distribution')
axes[2].set_xlabel('Characters')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.suptitle('CLA Pseudo Dataset Overview', y=1.02, fontsize=14, fontweight='bold')
plt.savefig('../results/dataset_overview.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Build Shared Vector Index

In [ ]:
from src.document_processor import DocumentProcessor
from src.vector_store import VectorStoreManager

processor = DocumentProcessor.from_config(CONFIG_PATH)
chunks = processor.process_dataset(DATASET_PATH)

print(f'Total chunks: {len(chunks)}')
print(f'Avg chunk length: {sum(len(c.text) for c in chunks) / len(chunks):.0f} chars')

# Show sample chunk
sample = chunks[0]
print(f'\nSample chunk ({sample.chunk_id}):')
print(sample.text[:300])

## 4. Knowledge Graph Inspection

In [ ]:
from src.knowledge_graph import KnowledgeGraphBuilder
from src.document_processor import DocumentLoader

documents = DocumentLoader.from_json(DATASET_PATH)
kg_builder = KnowledgeGraphBuilder()
kg_builder.build_from_documents(documents)

stats = kg_builder.get_stats()
print('Knowledge Graph Statistics:')
print(f'  Nodes: {stats["num_nodes"]}')
print(f'  Edges: {stats["num_edges"]}')
print(f'\n  Entity types: {stats["entity_type_counts"]}')
print(f'\n  Top entities by degree:')
for name, degree in stats['top_degree_nodes'][:10]:
    print(f'    {name:<30} degree={degree}')

In [ ]:
import networkx as nx

# Visualize subgraph around sirolimus
neighborhood = kg_builder.query_entity_neighborhood('sirolimus', depth=2)
print(f"Sirolimus neighborhood: {len(neighborhood['nodes'])} nodes, {len(neighborhood['edges'])} edges")

if neighborhood['edges']:
    sub_nodes = [n['name'] for n in neighborhood['nodes'][:15]]
    sub_edges = [(e['source'], e['target'], e['relation']) for e in neighborhood['edges'][:20]]
    
    G_sub = nx.DiGraph()
    G_sub.add_nodes_from(sub_nodes)
    G_sub.add_edges_from([(s, t) for s, t, _ in sub_edges])
    
    fig, ax = plt.subplots(figsize=(12, 8))
    pos = nx.spring_layout(G_sub, k=2, seed=42)
    
    color_map = {
        'sirolimus': '#FF6B6B',
    }
    node_colors = ['#FF6B6B' if n == 'sirolimus' else '#4ECDC4' for n in G_sub.nodes]
    node_sizes = [2000 if n == 'sirolimus' else 800 for n in G_sub.nodes]
    
    nx.draw_networkx_nodes(G_sub, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G_sub, pos, font_size=8, ax=ax)
    nx.draw_networkx_edges(G_sub, pos, arrows=True, arrowsize=15, width=1.5, 
                           edge_color='gray', alpha=0.7, ax=ax,
                           connectionstyle='arc3,rad=0.1')
    
    edge_labels = {(s, t): rel for s, t, rel in sub_edges if G_sub.has_edge(s, t)}
    nx.draw_networkx_edge_labels(G_sub, pos, edge_labels, font_size=6, ax=ax)
    
    ax.set_title('Knowledge Graph: Sirolimus Neighborhood (depth=2)', fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('../results/kg_sirolimus_neighborhood.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Single Question – Compare All Pipelines

In [ ]:
import importlib
from pipelines.base_pipeline import RAGResponse

# --- Load all pipelines ---
pipeline_configs = [
    ('Basic RAG',           'pipelines.01_basic_rag',           'BasicRAGPipeline'),
    ('KG-RAG',              'pipelines.02_knowledge_graph_rag', 'KnowledgeGraphRAGPipeline'),
    ('HyDE RAG',            'pipelines.03_hyde_rag',            'HyDERAGPipeline'),
    ('Self-RAG',            'pipelines.04_self_rag',            'SelfRAGPipeline'),
    ('Multi-hop RAG',       'pipelines.05_multihop_rag',        'MultiHopRAGPipeline'),
]

pipelines = {}
for name, module_path, class_name in pipeline_configs:
    mod = importlib.import_module(module_path)
    cls = getattr(mod, class_name)
    pipeline = cls(config_path=CONFIG_PATH)
    pipeline.build_index(DATASET_PATH)
    pipelines[name] = pipeline
    print(f'✓ {name} loaded')

print('\nAll pipelines ready!')

In [ ]:
TEST_QUESTION = "What genetic mutations are found in kaposiform lymphangiomatosis and what targeted therapies are available?"

print(f'Question: {TEST_QUESTION}\n')
print('=' * 70)

responses: dict[str, RAGResponse] = {}
for name, pipeline in pipelines.items():
    print(f'\n[{name}]')
    response = pipeline.timed_query(TEST_QUESTION, k=5)
    responses[name] = response
    print(f'Answer: {response.answer[:300]}...')
    print(f'Latency: {response.latency_seconds:.2f}s | Chunks: {len(response.retrieved_contexts)}')

In [ ]:
# Show reasoning traces for advanced pipelines
for name in ['Self-RAG', 'Multi-hop RAG']:
    if name in responses and responses[name].reasoning_trace:
        print(f'\n{"="*60}')
        print(f'{name} – Reasoning Trace:')
        for step in responses[name].reasoning_trace:
            print(f'  {step}')

## 6. Full Evaluation on All Eval Questions

In [ ]:
from src.evaluation import RAGEvaluator, RAGSample
from pathlib import Path

Path('../results').mkdir(exist_ok=True)

with open(EVAL_PATH) as f:
    eval_data = json.load(f)
eval_questions = eval_data['questions']

print(f'Evaluating on {len(eval_questions)} questions...')
print(f'Categories: {set(q["category"] for q in eval_questions)}')

In [ ]:
evaluator = RAGEvaluator(use_llm_judge=False)  # Set True + provide OpenAI key for LLM judge

all_summaries: dict[str, dict] = {}
all_results_df_rows = []

for pipeline_name, pipeline in pipelines.items():
    print(f'\nEvaluating {pipeline_name}...')
    samples = []
    latencies = []
    
    for qa in eval_questions:
        response = pipeline.timed_query(qa['question'], k=5)
        samples.append(RAGSample(
            question=qa['question'],
            answer=response.answer,
            contexts=response.retrieved_contexts,
            reference_answer=qa.get('reference_answer', ''),
            metadata={'category': qa['category'], 'difficulty': qa['difficulty']},
        ))
        latencies.append(response.latency_seconds)
        print(f'  Q: {qa["question"][:50]}... [{response.latency_seconds:.2f}s]')
    
    results = evaluator.evaluate_batch(samples)
    summary = evaluator.summarize(results)
    summary['avg_latency_s'] = round(sum(latencies) / len(latencies), 3)
    summary['pipeline'] = pipeline_name
    all_summaries[pipeline_name] = summary
    
    # Save per-pipeline results
    evaluator.save_results(results, f'../results/{pipeline_name.lower().replace(" ", "_")}_eval.json')
    
    for r, qa in zip(results, eval_questions):
        all_results_df_rows.append({
            'pipeline': pipeline_name,
            'question': r.question[:60],
            'category': qa['category'],
            'difficulty': qa['difficulty'],
            'context_precision': r.context_precision,
            'bert_f1': r.bert_f1,
            'rouge_l': r.rouge_l,
        })

print('\nEvaluation complete!')

In [ ]:
# Summary table
summary_df = pd.DataFrame(all_summaries).T
summary_df = summary_df.reset_index(drop=True)

cols = ['pipeline', 'context_precision', 'bert_f1', 'rouge_l', 'avg_latency_s']
display_df = summary_df[cols].copy()
display_df.columns = ['Pipeline', 'Context Precision', 'BERTScore F1', 'ROUGE-L', 'Avg Latency (s)']

print('\n=== Evaluation Summary ===')
print(display_df.to_string(index=False))

display_df.to_csv('../results/pipeline_comparison_summary.csv', index=False)
print('\nSaved to ../results/pipeline_comparison_summary.csv')

## 7. Visualization

In [ ]:
pipeline_names = [row['pipeline'] for row in all_summaries.values()]
colors = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0', '#F44336']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [
    ('context_precision', 'Context Precision', 'Higher = retrieved chunks are more relevant'),
    ('bert_f1', 'BERTScore F1', 'Higher = semantic similarity to reference answer'),
    ('avg_latency_s', 'Avg Latency (s)', 'Lower = faster response'),
]

for ax, (metric, title, subtitle) in zip(axes, metrics):
    values = [all_summaries[p].get(metric, 0) for p in pipeline_names]
    bars = ax.bar(pipeline_names, values, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{title}\n({subtitle})', fontsize=10)
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('RAG Pipeline Comparison – CLA Q&A Evaluation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/pipeline_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Radar chart: multi-dimensional comparison
from matplotlib.patches import FancyArrowPatch

categories = ['Context\nPrecision', 'BERTScore\nF1', 'ROUGE-L', 'Speed\n(1/latency)']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for i, (name, color) in enumerate(zip(pipeline_names, colors)):
    s = all_summaries[name]
    # Normalize latency to 0-1 (inverted, so higher = faster)
    max_lat = max(all_summaries[p].get('avg_latency_s', 1) for p in pipeline_names)
    speed = 1 - s.get('avg_latency_s', 1) / max_lat
    
    values_raw = [
        s.get('context_precision', 0),
        s.get('bert_f1', 0),
        s.get('rouge_l', 0),
        speed,
    ]
    values_raw += values_raw[:1]
    
    ax.plot(angles, values_raw, 'o-', linewidth=2, color=color, label=name)
    ax.fill(angles, values_raw, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
ax.set_title('RAG Pipeline Radar Chart\n(CLA Q&A Performance)', fontsize=13, pad=20, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/pipeline_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-category analysis
results_df = pd.DataFrame(all_results_df_rows)

if not results_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # By category
    cat_pivot = results_df.pivot_table(
        values='bert_f1', index='category', columns='pipeline', aggfunc='mean'
    )
    cat_pivot.plot(kind='bar', ax=axes[0], color=colors[:len(pipeline_names)], width=0.7)
    axes[0].set_title('BERTScore F1 by Question Category')
    axes[0].set_xlabel('Category')
    axes[0].set_ylabel('BERTScore F1')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend(fontsize=7)
    
    # By difficulty
    diff_pivot = results_df.pivot_table(
        values='bert_f1', index='difficulty', columns='pipeline', aggfunc='mean'
    )
    diff_pivot.plot(kind='bar', ax=axes[1], color=colors[:len(pipeline_names)], width=0.7)
    axes[1].set_title('BERTScore F1 by Question Difficulty')
    axes[1].set_xlabel('Difficulty')
    axes[1].set_ylabel('BERTScore F1')
    axes[1].tick_params(axis='x', rotation=0)
    axes[1].legend(fontsize=7)
    
    plt.tight_layout()
    plt.savefig('../results/pipeline_by_category.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. CLA-Specific Case Study: Multi-hop Question Analysis

In [ ]:
# Complex multi-hop question that requires chained reasoning
complex_q = (
    "Which CLA subtype is driven by TSC2 mutations, what drug is FDA-approved for it, "
    "and what was the key clinical trial that established this treatment?"
)

print(f'Complex Question:\n{complex_q}\n')
print('This requires:')
print('  Hop 1: TSC2 mutations → which disease?')
print('  Hop 2: That disease → which FDA-approved drug?')
print('  Hop 3: That drug → which clinical trial?')

print('\n' + '='*70)
for name in ['Basic RAG', 'Multi-hop RAG']:
    pipeline = pipelines[name]
    response = pipeline.timed_query(complex_q, k=5)
    print(f'\n[{name}] (latency: {response.latency_seconds:.2f}s)')
    print(f'Answer: {response.answer}')
    if response.reasoning_trace:
        print(f'\nReasoning ({len(response.reasoning_trace)} steps):')
        for step in response.reasoning_trace[:10]:
            print(f'  > {step}')

## 9. Summary & Recommendations

In [ ]:
print('=== FINAL PIPELINE RECOMMENDATIONS FOR CLA CHATBOT ===')
print()

recommendations = [
    ('Production deployment (speed priority)', 'Basic RAG', 
     'Lowest latency, sufficient for most Q&A'),
    ('Mechanism / relationship queries', 'Knowledge Graph RAG',
     'KG triples capture drug-target-pathway relationships'),
    ('High-precision clinical Q&A', 'HyDE RAG',
     'HyDE improves retrieval precision in specialized medical domains'),
    ('High-stakes / fact-critical answers', 'Self-RAG',
     'Self-reflection reduces hallucinations for safety-critical info'),
    ('Complex diagnostic/treatment planning', 'Multi-hop RAG',
     'Chains evidence across multiple retrieval steps'),
]

for use_case, pipeline, reason in recommendations:
    print(f'  {use_case}:')
    print(f'    → {pipeline}: {reason}')
    print()

print('Key Findings:')
print('  1. All pipelines benefit from the CLA-curated corpus (vs. general web)')
print('  2. KG-RAG excels on relationship questions (What does X treat? How does Y cause Z?)')
print('  3. Multi-hop RAG has highest latency but best on complex multi-part questions')
print('  4. Self-RAG provides the most consistent quality across difficulty levels')
print('  5. HyDE RAG bridges the semantic gap between lay questions and medical literature')